# PhysIR — Environment & hardware capture (Kaggle **and** Colab, no datasets needed)

Records everything §4.2 of the thesis needs: Python/OS, CPU/RAM, GPU(s), CUDA/cuDNN/driver, and
the **version of every package the pipeline actually imports**. Also profiles the model
(parameters + MACs) for the efficiency table, since that needs the code but **not** the datasets.

**Why this exists:** `requirements.txt` is inherited from DarkIR and was never used to build the
runtime — the training notebooks install *unpinned* on top of the platform image, and
`run_config.json` records no environment. This notebook closes that gap.

## How to run
- **Attach only `physir-code`.** No image datasets, no checkpoints.
- **Kaggle:** add the `physir-code` dataset → *Run All*. Accelerator **GPU T4 × 2** so the
  reported GPU matches the 256-px regime.
- **Colab:** needs the `COLAB_TOKEN` secret (same as the training notebooks) to fetch the code
  from Kaggle; runs fine without it, only the model-profile section is skipped. Pick the
  **A100** runtime if any credit remains, so the reported GPU matches the 384-px regime — a few
  minutes only. *(If not, any GPU runtime still captures the identical software stack; only the
  device name differs, and that is already documented from the run records.)*

Run it **once per platform**. Output: a markdown block ready to paste into §4.2, plus
`env_capture_<platform>.json` in the working directory.

## 1. Install (identical to the training notebooks)

In [ ]:
# Same install line as the training notebooks — so captured versions match what training used.
!pip install -q lpips pytorch-msssim einops ptflops pyiqa 2>/dev/null
print('install step done')

## 2. Platform detection + locate the code (no datasets required)

In [ ]:
import os, sys, glob, json, platform, subprocess, shutil

def detect():
    if os.path.isdir('/kaggle/input'): return 'kaggle'
    try:
        import google.colab  # noqa: F401
        return 'colab'
    except ImportError:
        return 'local'
PLATFORM = detect()
print('platform:', PLATFORM)

CODE_ROOT = None
def _find(base):
    hits = sorted(glob.glob(os.path.join(base, '**', 'train.py'), recursive=True))
    return os.path.dirname(hits[0]) if hits else None

if PLATFORM == 'kaggle':
    for d in sorted(glob.glob('/kaggle/input/*')):
        CODE_ROOT = CODE_ROOT or _find(d)
elif PLATFORM == 'colab':
    dest = '/content/physir_code'
    if not os.path.isdir(dest):
        try:                                    # same auth path as the training notebooks
            from google.colab import userdata
            os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
            tok = userdata.get('COLAB_TOKEN')
            with open(os.path.expanduser('~/.kaggle/access_token'), 'w') as f: f.write(tok)
            if tok.lstrip().startswith('{'):
                with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f: f.write(tok)
                os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
            os.makedirs(dest, exist_ok=True)
            os.system(f'pip install -q kaggle && kaggle datasets download '
                      f'-d jeffreyboahen/physir-code --unzip -p {dest} -q')
        except Exception as e:
            print(f'  (could not fetch code: {e} — model profile will be skipped)')
    CODE_ROOT = _find(dest)
else:
    CODE_ROOT = _find(os.getcwd())

if CODE_ROOT and CODE_ROOT not in sys.path:
    sys.path.insert(0, CODE_ROOT)
print('code root:', CODE_ROOT or 'NOT FOUND — environment capture still runs, model profile skipped')

## 3. Environment capture

In [ ]:
import importlib
env = {'platform': PLATFORM}

# ── interpreter / OS ───────────────────────────────────────────────────────
env['python']   = sys.version.split()[0]
env['os']       = platform.platform()
env['machine']  = platform.machine()

# ── CPU / RAM ──────────────────────────────────────────────────────────────
try:
    env['cpu_count'] = os.cpu_count()
    with open('/proc/meminfo') as f:
        for line in f:
            if line.startswith('MemTotal'):
                env['ram_gb'] = round(int(line.split()[1]) / 1048576, 1); break
    for line in open('/proc/cpuinfo'):
        if line.startswith('model name'):
            env['cpu_model'] = line.split(':', 1)[1].strip(); break
except Exception:
    pass

# ── GPU / CUDA ─────────────────────────────────────────────────────────────
import torch
env['torch_cuda_build'] = torch.version.cuda
env['cudnn'] = torch.backends.cudnn.version()
env['gpu_count'] = torch.cuda.device_count()
env['gpus'] = []
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    env['gpus'].append({'name': p.name, 'total_mem_gb': round(p.total_memory / 2**30, 1),
                        'capability': f'{p.major}.{p.minor}', 'multiprocessors': p.multi_processor_count})
try:
    env['nvidia_driver'] = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'],
        text=True).strip().splitlines()[0]
except Exception:
    env['nvidia_driver'] = 'n/a'

# ── package versions (exactly what the pipeline imports) ───────────────────
# module name -> pip distribution name (several packages expose no __version__ attribute,
# so fall back to the installed distribution metadata rather than reporting 'unknown')
PKGS = {'torch': 'torch', 'torchvision': 'torchvision', 'numpy': 'numpy', 'PIL': 'pillow',
        'einops': 'einops', 'pytorch_msssim': 'pytorch-msssim', 'ptflops': 'ptflops',
        'yaml': 'PyYAML', 'tqdm': 'tqdm', 'lpips': 'lpips', 'pyiqa': 'pyiqa',
        'pandas': 'pandas', 'matplotlib': 'matplotlib'}
try:
    from importlib.metadata import version as _dist_version, PackageNotFoundError
except ImportError:
    from importlib_metadata import version as _dist_version, PackageNotFoundError

env['packages'] = {}
for mod, dist in PKGS.items():
    v = None
    try:
        v = getattr(importlib.import_module(mod), '__version__', None)
    except Exception:
        env['packages'][mod] = 'NOT INSTALLED'; continue
    if not v:
        try: v = _dist_version(dist)
        except PackageNotFoundError: v = 'unknown'
    env['packages'][mod] = v

for k in ('python', 'os', 'cpu_model', 'ram_gb', 'torch_cuda_build', 'cudnn', 'nvidia_driver'):
    if k in env: print(f'{k:18s} {env[k]}')
for g in env['gpus']:
    print(f'{"gpu":18s} {g["name"]}  {g["total_mem_gb"]} GB  sm_{g["capability"].replace(".","")}')
print()
for k, v in env['packages'].items():
    print(f'  {k:16s} {v}')

## 4. Model profile — parameters and MACs (efficiency table)

In [ ]:
# Model profile — needs the code, NOT the datasets. Feeds the efficiency table (T-G).
env['model'] = {}
if CODE_ROOT:
    try:
        # use the project's own builder (it maps 'img_channels' -> the constructor's
        # 'img_channel'), so this matches exactly how every training notebook builds the model
        import train as T
        from ptflops import get_model_complexity_info
        NET = {'name': 'DarkIR', 'img_channels': 3, 'width': 32,
               'middle_blk_num_enc': 2, 'middle_blk_num_dec': 2,
               'enc_blk_nums': [1, 2, 3], 'dec_blk_nums': [3, 1, 1],
               'dilations': [1, 4, 9], 'extra_depth_wise': True}
        model = T.build_model(NET).eval()
        n_par = sum(p.numel() for p in model.parameters())
        env['model']['params_M'] = round(n_par / 1e6, 3)
        env['model']['params_trainable_M'] = round(
            sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6, 3)
        for res in ((256, 256), (384, 384), (640, 1120)):
            try:
                macs, _ = get_model_complexity_info(
                    model, (3, *res), as_strings=False, print_per_layer_stat=False, verbose=False)
                env['model'][f'GMACs_{res[1]}x{res[0]}'] = round(macs / 1e9, 2)
            except Exception as e:
                env['model'][f'GMACs_{res[1]}x{res[0]}'] = f'failed: {e}'
        # the physics losses add exactly one trainable scalar, and only during training
        try:
            from losses import create_physics_loss
            pl, extra = create_physics_loss(
                {'retinex': True, 'blur': True, 'phase': True}, 'cpu')
            env['model']['physics_extra_trainable_params'] = int(
                sum(p.numel() for p in extra))
            env['model']['physics_terms'] = sorted(pl)
        except Exception as e:
            env['model']['physics_note'] = f'loss import failed: {e}'
        for k, v in env['model'].items():
            print(f'  {k:34s} {v}')
        print('\n  → identical at inference to unmodified DarkIR-m: the physics terms are '
              'training-only\n    and contribute no forward-path parameters or MACs.')
    except Exception as e:
        print('model profile skipped:', e)
else:
    print('no code root — model profile skipped')

## 5. Emit a paste-ready block for §4.2

In [ ]:
out = f'/kaggle/working/env_capture_{PLATFORM}.json' if PLATFORM == 'kaggle' else f'env_capture_{PLATFORM}.json'
json.dump(env, open(out, 'w'), indent=2)

g = env['gpus'][0] if env['gpus'] else None
gpu_line = (f"{env['gpu_count']} × {g['name']} ({g['total_mem_gb']} GB)" if g else 'CPU only')
pk = env['packages']
print(f'''
=========== paste into §4.2 ===========

**{PLATFORM.capitalize()} environment.** {gpu_line}; NVIDIA driver {env.get('nvidia_driver','n/a')},
CUDA {env['torch_cuda_build']}, cuDNN {env['cudnn']}. Host: {env.get('cpu_model','n/a')},
{env.get('cpu_count','?')} vCPU, {env.get('ram_gb','?')} GB RAM. Python {env['python']} on {env['os']}.

| package | version |   | package | version |
|---|---|---|---|---|
| torch | {pk['torch']} |  | lpips | {pk['lpips']} |
| torchvision | {pk['torchvision']} |  | pyiqa | {pk['pyiqa']} |
| numpy | {pk['numpy']} |  | pillow | {pk['PIL']} |
| einops | {pk['einops']} |  | PyYAML | {pk['yaml']} |
| pytorch-msssim | {pk['pytorch_msssim']} |  | tqdm | {pk['tqdm']} |
| ptflops | {pk['ptflops']} |  | pandas | {pk['pandas']} |

=======================================
saved: {out}''')